### Connection Mysql

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [3]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [4]:
import pandas as pd
import json

In [5]:
import re

def sql_like_to_regex(pattern: str) -> str:                                                                   
     escaped = re.escape(pattern)
     regex = escaped.replace('%', '.*').replace('_', '.')                       
     return f'^{regex}$'                                                        
                                                                                

def like(series: pd.Series, pattern: str, case: bool = True) -> pd.Series:     
     return series.str.contains(                                                
         sql_like_to_regex(pattern), case=case, regex=True, na=False            
     )   

### Function Extract Details Dosen

In [6]:
def parse_details(x):
  if pd.isna(x):
    return {}
  try:
    data = json.loads(x)
    for key in ['serdos', 'status_pegawai', 'jabfung', 'gelar_depan', 'gelar_belakang']:
      data.pop(key, None)
    return data
  except Exception:
    return {}

### Reference Jenjang Pendidikan Dosen

In [7]:
jenjangPendidikanQuery = text("""
SELECT code, name FROM jenjang_pendidikan
""")

jenjangPendidikanDf = pd.read_sql(jenjangPendidikanQuery, siakadConnection)
jenjangPendidikanDf['code'] = jenjangPendidikanDf['code'].astype(str)
jenjangPendidikanDf = jenjangPendidikanDf.rename(columns={
    'name': 'jenjang_pendidikan'
})

### Dosen

In [8]:
query = text("""
SELECT DISTINCT
	cp.`name` AS prodi,
	kl.semester_code AS semester_kelas,
	s.year_of_study,
	m.`name` AS matakuliah,
	d.name,
	d.details
FROM
	curriculums AS c -- Kurikulum (c)
	JOIN prodi AS cp ON c.prodi_uuid = cp.uuid -- Kurikulum Prodi (cp)
	JOIN curriculum_courses AS cc ON cc.curriculum_uuid = c.uuid AND cc.deleted_at IS NULL AND cc.type = 'wajib'  -- Matakuliah Kurikulum (cc)
	JOIN matakuliah AS m ON cc.matkul_uuid = m.uuid AND m.deleted_at IS NULL -- Matakuliah (m)
	JOIN kelas_laporan AS kl ON kl.matkul_uuid = m.uuid AND kl.deleted_at IS NULL -- Kelas Laporan (kl)
	JOIN semesters AS s ON kl.semester_code = s.`code` AND s.year_of_study >= (YEAR (CURDATE()) - 4 ) -- Semester (s)
	JOIN dosen_kelas_laporan AS dkl ON dkl.kelas_uuid = kl.uuid AND dkl.deleted_at IS NULL -- Dosen Kelas Laporan (dkl)
	JOIN dosen_list AS d ON dkl.dosen_uuid = d.uuid AND d.deleted_at IS NULL -- Dosen (d)
WHERE
	c.deleted_at IS NULL
""")

In [9]:
df = pd.read_sql(query, siakadEngine)

### Normalize Dosen Details

In [10]:
details = pd.json_normalize(df['details'].apply(parse_details))

In [11]:
details = details.rename(columns={
    'pendidikan_terakhir': 'code'
})

In [12]:
details = details.merge(
    jenjangPendidikanDf,
    left_on='code',
    right_on='code',
    how='left'
).drop(columns=['code'])

In [13]:
df = pd.concat([df.drop(columns=['details']), details], axis=1)

In [14]:
df.to_csv('data/dosen/dosen-dtps.csv', index=False)